# 📈 Notebook 1: Understanding Write Bottlenecks

Before scaling, we need to understand what makes writes different from reads and identify where bottlenecks occur.

## Learning Objectives

By the end of this notebook, you'll understand:
- How writes differ from reads
- Where write bottlenecks occur
- How to measure write throughput
- When write scaling is actually needed

## 🛠️ Setup

Before running the code cells, make sure the lab's services and Python environment are ready:

```bash
cd 04-patterns/scaling-writes

# 1. Start PostgreSQL + Redis + Adminer + RedisInsight
docker compose up -d

# 2. Install Python dependencies into a per-lab .venv (managed by uv)
uv sync
```

Then, in **VS Code**:

1. Open this notebook.
2. Click the **kernel picker** in the top-right of the notebook.
3. Choose the `.venv` interpreter for this lab (`04-patterns/scaling-writes/.venv`).
4. If the kernel doesn't show up, press `Cmd+Shift+P` → **"Reload Window"** and try again.

🔍 **Open Adminer** at http://localhost:8080 to watch write operations!
   Server: `postgres`, User: `demo`, Password: `demo`, Database: `writes_demo`.


In [ ]:
import psycopg2
import time
import statistics
from concurrent.futures import ThreadPoolExecutor

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "writes_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

try:
    conn = get_connection()
    print("✅ Connected to PostgreSQL")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker compose up -d")

## 📊 Reads vs Writes: The Fundamental Difference

In [ ]:
print("📊 Reads vs Writes: What Makes Them Different")
print("=" * 60)
print("""
READS:
─────────────────────────────────────────────────────────────
• Can be served from cache/memory
• Can be parallelized across replicas
• Stateless - doesn't change data
• Failure = retry (no side effects)

WRITES:
─────────────────────────────────────────────────────────────
• Must go to disk (durability)
• Must go to single leader (consistency)
• Stateful - changes data permanently
• Failure = complex (partial writes?)
• Requires: locks, indexes, replication

THE ASYMMETRY:
─────────────────────────────────────────────────────────────
• Adding read replicas: Easy! Just copy data.
• Adding write capacity: Hard! Must coordinate.

This is why write scaling requires different strategies!
""")

## 🔬 Measuring Write Throughput

In [ ]:
def single_write() -> float:
    conn = get_connection()
    cursor = conn.cursor()
    
    start = time.time()
    cursor.execute(
        "INSERT INTO events (event_type, user_id, payload) VALUES (%s, %s, %s)",
        ('click', 1, '{"page": "home"}')
    )
    conn.commit()
    elapsed = time.time() - start
    
    conn.close()
    return elapsed * 1000

print("🔬 Measuring Single Write Latency")
print("=" * 60)

times = [single_write() for _ in range(100)]

print(f"\nSingle write statistics (100 writes):")
print(f"   Mean:   {statistics.mean(times):.2f}ms")
print(f"   Median: {statistics.median(times):.2f}ms")
# p95 = the value below which 95% of samples fall.
# For 100 samples that's index ceil(0.95*100)-1 = 94 in a 0-indexed sorted list.
p95 = sorted(times)[int(0.95 * len(times)) - 1]
print(f"   P95:    {p95:.2f}ms")
print(f"   Max:    {max(times):.2f}ms")

writes_per_sec = 1000 / statistics.mean(times)
print(f"\n📊 Estimated throughput: ~{writes_per_sec:.0f} writes/sec (single thread)")

# Sanity-check the sample and the percentile index. A p95 below the median
# means the index arithmetic above drifted.
assert len(times) == 100, f"expected 100 samples, got {len(times)}"
assert p95 >= statistics.median(times), (
    f"p95 ({p95:.2f}ms) cannot be below the median "
    f"({statistics.median(times):.2f}ms) -- check the percentile index"
)


In [ ]:
def write_worker(num_writes: int) -> list:
    times = []
    conn = get_connection()
    cursor = conn.cursor()
    
    for i in range(num_writes):
        start = time.time()
        cursor.execute(
            "INSERT INTO events (event_type, user_id, payload) VALUES (%s, %s, %s)",
            ('click', i, '{"page": "home"}')
        )
        conn.commit()
        times.append((time.time() - start) * 1000)
    
    conn.close()
    return times

print("🔬 Concurrent Write Throughput Test")
print("=" * 60)

throughput = {}

for num_workers in [1, 5, 10, 20]:
    writes_per_worker = 50
    
    start = time.time()
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(write_worker, writes_per_worker) for _ in range(num_workers)]
        all_times = []
        for f in futures:
            all_times.extend(f.result())
    total_time = time.time() - start
    
    total_writes = num_workers * writes_per_worker
    throughput[num_workers] = total_writes / total_time
    
    print(f"\n👥 {num_workers} concurrent writers:")
    print(f"   Total writes: {total_writes}")
    print(f"   Total time:   {total_time:.2f}s")
    print(f"   Throughput:   {throughput[num_workers]:.0f} writes/sec")
    print(f"   Avg latency:  {statistics.mean(all_times):.2f}ms")

# Notice what did NOT happen: latency stayed roughly flat while throughput
# climbed. Each worker commits every row, so it spends most of its time waiting
# on a round-trip + fsync; extra writers overlap that wait rather than fight
# for CPU. That is why "one connection's throughput" is never your ceiling.
speedup = throughput[20] / throughput[1]
print(f"\n📈 20 writers reached {speedup:.1f}x the throughput of 1 writer.")

# If this ever fails, the machine (or the DB) has become the bottleneck before
# concurrency can help, and every number printed above is measuring something
# other than what this section claims to teach.
assert speedup > 1.5, (
    f"expected 20 concurrent writers to beat a single writer by >1.5x, got "
    f"{speedup:.1f}x ({throughput[20]:.0f} vs {throughput[1]:.0f} writes/sec)"
)


## 🎯 Write Bottleneck Sources

In [ ]:
print("🎯 Where Write Bottlenecks Occur")
print("=" * 60)
print("""
1. DISK I/O
─────────────────────────────────────────────────────────────
   • Write-ahead log (WAL) must be flushed
   • Data pages written to disk
   • SSD: ~100,000 IOPS | HDD: ~100 IOPS

2. CPU
─────────────────────────────────────────────────────────────
   • Query parsing and planning
   • Index updates (B-tree rebalancing)
   • Constraint checking
   • Triggers and stored procedures

3. MEMORY
─────────────────────────────────────────────────────────────
   • Buffer pool for dirty pages
   • Lock management
   • Connection overhead

4. NETWORK
─────────────────────────────────────────────────────────────
   • Replication to replicas
   • Client-server round trips
   • Distributed transactions

5. CONTENTION
─────────────────────────────────────────────────────────────
   • Row locks on same data
   • Table locks
   • Index page locks
""")

In [ ]:
print("📈 Index Overhead on Writes")
print("=" * 60)

conn = get_connection()
cursor = conn.cursor()
cursor.execute("SELECT indexname FROM pg_indexes WHERE tablename = 'events'")
indexes = [row[0] for row in cursor.fetchall()]
conn.close()

print(f"\nCurrent indexes on 'events' table:")
for idx in indexes:
    print(f"   • {idx}")

# db/init.sql seeds a primary key plus three secondary indexes. Every INSERT
# into `events` therefore touches 4 B-tree structures, not just the heap.
assert len(indexes) >= 4, (
    f"expected the primary key plus the 3 seeded indexes from db/init.sql, "
    f"found {indexes} -- did the container start with a stale volume?"
)

print(f"\n💡 Each index must be updated on every INSERT!")
print(f"   {len(indexes)} indexes = {len(indexes)} B-trees to update on top of")
print(f"   the heap write itself, every single row.")
print(f"   More indexes = slower writes, faster reads.")
print(f"   This is the classic read/write trade-off.")
print(f"\n   We only *count* them here. Notebook 2 measures what they cost:")
print(f"   it builds two identical tables (PK-only vs +5 indexes) and times")
print(f"   the same insert against both.")


## 🧮 Back-of-Envelope: Do You Need Write Scaling?

In [ ]:
print("🧮 Back-of-Envelope Calculation")
print("=" * 60)
print("""
SCENARIO: Social media like button
─────────────────────────────────────────────────────────────

Given:
• 100 million daily active users
• Average user likes 10 posts/day
• Peak traffic = 3x average

Calculation:
• Daily likes = 100M × 10 = 1 billion likes/day
• Average per second = 1B / 86,400 ≈ 11,500 likes/sec
• Peak = 11,500 × 3 ≈ 35,000 likes/sec

Can a single PostgreSQL handle this?
─────────────────────────────────────────────────────────────
• Well-tuned PostgreSQL: ~10,000-50,000 writes/sec
• Answer: Maybe at average, NO at peak!

We need write scaling strategies!
""")

def calculate_write_needs(dau: int, actions_per_user: int, peak_multiplier: float = 3.0):
    daily_writes = dau * actions_per_user
    avg_per_second = daily_writes / 86400
    peak_per_second = avg_per_second * peak_multiplier
    
    print(f"\n📊 Your scenario:")
    print(f"   Daily writes:    {daily_writes:,.0f}")
    print(f"   Average/sec:     {avg_per_second:,.0f}")
    print(f"   Peak/sec:        {peak_per_second:,.0f}")
    
    if peak_per_second < 1000:
        print(f"   ✅ Single DB can handle this easily")
    elif peak_per_second < 10000:
        print(f"   ⚠️ Need optimization, maybe single DB")
    else:
        print(f"   ❌ Need sharding/batching strategies")

calculate_write_needs(dau=1_000_000, actions_per_user=5)
calculate_write_needs(dau=100_000_000, actions_per_user=10)

## 🧪 Quick Quiz

1. **Why can't you just add write replicas like read replicas?**

2. **What happens to write throughput as you add more indexes?**

3. **When should you NOT worry about write scaling?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Why no write replicas:")
print("   - Writes must be coordinated (consistency)")
print("   - Multi-master = conflict resolution nightmare")
print("   - Read replicas receive writes FROM leader")
print()
print("2. More indexes = slower writes:")
print("   - Each INSERT updates ALL indexes")
print("   - B-tree rebalancing has overhead")
print("   - Classic read/write trade-off")
print()
print("3. When NOT to worry:")
print("   - < 1,000 writes/sec (most apps!)")
print("   - When reads are the actual bottleneck")
print("   - Before you've done the math!")

## 📚 Summary

### Key Takeaways

1. **Writes are harder to scale than reads** - Must coordinate, can't just replicate
2. **Do the math first** - Many apps don't need write scaling
3. **Bottlenecks vary** - Disk I/O, CPU, contention all matter
4. **Indexes help reads, hurt writes** - Know your trade-offs
5. **Peak matters more than average** - Design for bursts

### Next Up

In **Notebook 2**, we'll learn database optimization for writes:
- Write-optimized databases
- Index management
- WAL tuning